# DG-Hetero-GNN Paper Minimal-Diff Recovery

The attached paper is the sole normative specification. Commit 9 supplies only the recovered processed archives. The complete reviewed implementation is embedded below and is the code this notebook executes.

## Minimal-diff executable implementation

The reviewed engine remains in one contiguous cell to minimize notebook-level restructuring.

In [ ]:
"""Executable, leakage-safe DG-Hetero-GNN paper reproduction protocol.

The module deliberately contains no reported paper metrics.  All tables and
figures are rendered solely from prediction and result artifacts generated by
``run_suite``.
"""
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
import tempfile
import time
import zipfile
from datetime import datetime, timezone
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import sklearn
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data, HeteroData
from torch_geometric.nn import GATConv, GCNConv, HeteroConv, SAGEConv


HASHES = {
    "elliptic": "0a86c749a1c388be50ca2a828485532216dd1cf29f39e9a5a69159c66514ee29",
    "ieee": "e73460dffa01a807cdf113a768c535f8d21d9bf6a94e041ca863449d220bf761",
    "dgraphfin": "c7726f1dfd085e548ac86ad1cae62d751c87df95248260499ab431b9bfbb9e21",
    "amlsim": "99dd1eb32ad2a71fd6eb95eaa942951599f88418c1fafe884957a143491acb1e",
}
ARCHIVES = {"elliptic": "elliptic_standardized.zip", "ieee": "ieee_standardized.zip",
            "dgraphfin": "dgraphfin_reduced.zip", "amlsim": "amlsim_final_unified.zip"}
TABLE_1 = {"elliptic": (46564, 93128, 128, 222880), "ieee": (590540, 14962, 128, 2952700),
           "dgraphfin": (250000, 324822, 128, 1036164), "amlsim": (45, 35, 128, 443)}
DOMAIN_ID = {"elliptic": 0, "ieee": 1, "dgraphfin": 2, "amlsim": 3}
EXPECTED_EDGE_TYPES = {
    ("account", "makes", "transaction"),
    ("transaction", "made_by", "account"),
    ("transaction", "to", "merchant"),
    ("merchant", "receives", "transaction"),
    ("transaction", "interacts", "transaction"),
}
SEEDS = [42, 123, 3407, 2025, 9999]
LODO = {"LODO0": (["ieee", "dgraphfin"], "elliptic"),
        "LODO1": (["elliptic", "dgraphfin"], "ieee"),
        "LODO2": (["elliptic", "ieee"], "dgraphfin")}


@dataclass(frozen=True)
class Config:
    run_mode: str = "SMOKE_TEST"
    epochs: int = 40
    smoke_epochs: int = 2
    smoke_nodes: int = 512
    hidden: int = 48
    dropout: float = .15
    learning_rate: float = .0005
    weight_decay: float = 0.0
    focal_alpha: float = .25
    focal_gamma: float = 2.0
    domain_weight: float = .10
    grl_strength: float = 1.0
    validation_fraction: float = .20
    ece_bins: int = 10
    seeds: tuple = tuple(SEEDS)


def project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "recovery-source" / "Datasets").exists():
            return candidate
    raise FileNotFoundError("recovery-source/Datasets is required beside the notebooks")


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, default=_json_default), encoding="utf-8")
    temp.replace(path)


def _json_default(x):
    if isinstance(x, np.generic): return x.item()
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, Path): return str(x)
    raise TypeError(type(x).__name__)


def atomic_torch(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temp)
    temp.replace(path)


def _safe_extract_one(archive: Path) -> Path:
    with tempfile.TemporaryDirectory() as temporary:
        destination = Path(temporary).resolve()
        with zipfile.ZipFile(archive) as z:
            candidates = [i for i in z.infolist() if i.filename.endswith(".pt")]
            if len(candidates) != 1: raise ValueError(f"{archive.name} must have exactly one .pt")
            item = candidates[0]
            output = (destination / item.filename).resolve()
            if destination not in output.parents: raise ValueError("unsafe archive member")
            output.parent.mkdir(parents=True, exist_ok=True)
            with z.open(item) as source, output.open("wb") as sink: shutil.copyfileobj(source, sink)
            # torch.load must occur while the temporary file still exists.
            obj = torch.load(output, weights_only=False, map_location="cpu")
    return obj


def load_graph(name: str, dataset_dir: Path) -> HeteroData:
    archive = dataset_dir / ARCHIVES[name]
    if sha256(archive) != HASHES[name]: raise ValueError(f"SHA-256 mismatch: {archive}")
    graph = _safe_extract_one(archive)
    if isinstance(graph, list):
        if len(graph) != 1: raise ValueError(f"{name}: list serialization must contain one HeteroData")
        graph = graph[0]
    if not isinstance(graph, HeteroData): raise TypeError(f"{name}: archive did not contain HeteroData")
    for node_type in ("account", "merchant"):
        x = graph[node_type].x
        if x.size(1) == 32:
            graph[node_type].x = torch.cat((x, torch.zeros(x.size(0), 32, dtype=x.dtype)), dim=1)
        if graph[node_type].x.size(1) != 64: raise ValueError(f"{name}: {node_type} feature width")
    if set(graph.edge_types) != EXPECTED_EDGE_TYPES:
        raise ValueError(f"{name}: archived relation schema changed: {set(graph.edge_types)}")
    return graph


def graph_statistics(name: str, graph: HeteroData) -> dict:
    for node_type in ("transaction", "account", "merchant"):
        assert torch.isfinite(graph[node_type].x).all(), f"{name}/{node_type}: nonfinite x"
    labels = graph["transaction"].y.reshape(-1)
    assert set(labels.unique().tolist()) <= {0, 1}, f"{name}: labels are not binary"
    for edge_type in graph.edge_types:
        edge_index = graph[edge_type].edge_index
        assert edge_index.ndim == 2 and edge_index.size(0) == 2
        assert ((edge_index[0] >= 0) & (edge_index[0] < graph[edge_type[0]].num_nodes)).all()
        assert ((edge_index[1] >= 0) & (edge_index[1] < graph[edge_type[2]].num_nodes)).all()
    transactions = graph["transaction"].num_nodes
    entities = graph["account"].num_nodes + graph["merchant"].num_nodes
    archived_edges = sum(graph[e].edge_index.size(1) for e in graph.edge_types)
    assert (transactions, entities, 128, archived_edges) == TABLE_1[name]
    if name == "elliptic": assert int((labels == 0).sum()) == 42019 and int((labels == 1).sum()) == 4545
    return {"Dataset": name, "Transaction nodes": transactions, "Entity nodes": entities,
            "Feature size": 128, "Total edges": archived_edges}


def load_all(root: Path | None = None) -> tuple[dict[str, HeteroData], pd.DataFrame]:
    root = project_root(root)
    graphs = {name: load_graph(name, root / "recovery-source" / "Datasets") for name in ARCHIVES}
    table = pd.DataFrame([graph_statistics(name, graph) for name, graph in graphs.items()])
    return graphs, table


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.benchmark = False


class _GRL(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, strength): ctx.strength = strength; return input.view_as(input)
    @staticmethod
    def backward(ctx, output): return -ctx.strength * output, None


def grl(x: torch.Tensor, strength: float) -> torch.Tensor: return _GRL.apply(x, strength)


class FocalLoss(nn.Module):
    def __init__(self, alpha: float, gamma: float): super().__init__(); self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, labels):
        labels = labels.float(); bce = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        pt = torch.exp(-bce); alpha = torch.where(labels == 1, self.alpha, 1 - self.alpha)
        return (alpha * (1 - pt).pow(self.gamma) * bce).mean()


class DGHeteroGNN(nn.Module):
    def __init__(self, metadata, cfg: Config, *, n_domains=2, type_projection=True, message_passing=True, deep_head=True):
        super().__init__(); self.message_passing = message_passing; self.type_projection = type_projection; h = cfg.hidden
        if type_projection:
            self.encoder = nn.ModuleDict({"transaction": _projection(128, h, cfg.dropout),
                "account": _projection(64, h, cfg.dropout), "merchant": _projection(64, h, cfg.dropout)})
        else: self.encoder = _projection(128, h, cfg.dropout)
        self.layers = nn.ModuleList([HeteroConv({edge: SAGEConv((h, h), h) for edge in metadata[1]}, aggr="sum") for _ in range(3)])
        self.classifier = nn.Sequential(nn.Linear(h, 64), nn.ReLU(), nn.Dropout(.2), nn.Linear(64, 32), nn.ReLU(), nn.Dropout(.15), nn.Linear(32, 1)) if deep_head else nn.Linear(h, 1)
        self.domain_classifier = nn.Sequential(nn.Linear(h, 16), nn.ReLU(), nn.Dropout(.1), nn.Linear(16, n_domains))
    def forward(self, x_dict, edge_index_dict, grl_strength=1.0):
        if self.type_projection: x = {t: self.encoder[t](value.float()) for t, value in x_dict.items()}
        else: x = {t: self.encoder(F.pad(value.float(), (0, 128 - value.size(1)))) for t, value in x_dict.items()}
        if self.message_passing:
            for layer in self.layers: x = {t: F.relu(v) for t, v in layer(x, edge_index_dict).items()}
        tx = x["transaction"]
        return self.classifier(tx).flatten(), self.domain_classifier(grl(tx, grl_strength))


def _projection(input_dim, hidden, dropout): return nn.Sequential(nn.Linear(input_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(), nn.Dropout(dropout))


class HeteroGraphSAGE(nn.Module):
    """Independent heterogeneous baseline: no GRL and no domain classifier."""
    def __init__(self, metadata, cfg: Config):
        super().__init__(); h = cfg.hidden
        self.encoder = nn.ModuleDict({"transaction": _projection(128, h, cfg.dropout),
            "account": _projection(64, h, cfg.dropout), "merchant": _projection(64, h, cfg.dropout)})
        self.layers = nn.ModuleList([HeteroConv({edge: SAGEConv((h, h), h) for edge in metadata[1]}, aggr="sum") for _ in range(3)])
        self.classifier = nn.Linear(h, 1)
    def forward(self, x_dict, edge_index_dict):
        x = {t: self.encoder[t](value.float()) for t, value in x_dict.items()}
        for layer in self.layers: x = {t: F.relu(v) for t, v in layer(x, edge_index_dict).items()}
        return self.classifier(x["transaction"]).flatten()


class HomogeneousGNN(nn.Module):
    def __init__(self, family: str, hidden: int):
        super().__init__(); conv = {"gcn": GCNConv, "sage": SAGEConv, "gat": GATConv}[family]
        self.layers = nn.ModuleList([conv(128 if n == 0 else hidden, hidden) for n in range(3)])
        self.out = nn.Linear(hidden, 1)
    def forward(self, x, edge_index, txn_count):
        for layer in self.layers: x = F.relu(layer(x, edge_index))
        return self.out(x[:txn_count]).flatten()


class MLP(nn.Module):
    def __init__(self): super().__init__(); self.net = nn.Sequential(nn.Linear(128, 48), nn.ReLU(), nn.Linear(48, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x): return self.net(x).flatten()


def transaction_split(graph: HeteroData, seed: int, fraction: float) -> tuple[torch.Tensor, torch.Tensor]:
    y = graph["transaction"].y.cpu().numpy(); indices = np.arange(y.size)
    train, validation = train_test_split(indices, test_size=fraction, stratify=y, random_state=seed)
    assert not set(train).intersection(validation)
    return torch.as_tensor(train), torch.as_tensor(validation)


def _sample_graph(graph: HeteroData, seed: int, count: int) -> HeteroData:
    """Deterministic stratified induced subgraph for smoke runs only."""
    labels = graph["transaction"].y.cpu().numpy(); rng = np.random.default_rng(seed)
    chosen = []
    for label in (0, 1):
        candidates = np.flatnonzero(labels == label); take = min(len(candidates), max(1, count // 2))
        chosen.extend(rng.choice(candidates, take, replace=False).tolist())
    chosen = torch.tensor(sorted(set(chosen)))
    accounts, merchants = [], []
    for edge_type in graph.edge_types:
        if edge_type[1].startswith("rev_"): continue
        edge = graph[edge_type].edge_index
        if edge_type[0] == "account" and edge_type[2] == "transaction": accounts.extend(edge[0, torch.isin(edge[1], chosen)].tolist())
        if edge_type[0] == "transaction" and edge_type[2] == "merchant": merchants.extend(edge[1, torch.isin(edge[0], chosen)].tolist())
    subsets = {"transaction": chosen, "account": torch.tensor(sorted(set(accounts)), dtype=torch.long),
               "merchant": torch.tensor(sorted(set(merchants)), dtype=torch.long)}
    return graph.subgraph(subsets)


def _device(): return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _device_view(graph: HeteroData, device, *, include_labels: bool) -> HeteroData:
    """Create a new device view without mutating the authoritative CPU graph."""
    view = HeteroData()
    for node_type in graph.node_types:
        view[node_type].x = graph[node_type].x.to(device)
        if include_labels and node_type == "transaction":
            view[node_type].y = graph[node_type].y.to(device)
    for edge_type in graph.edge_types:
        view[edge_type].edge_index = graph[edge_type].edge_index.to(device)
    return view


def _progress(message: str) -> None:
    stamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    gpu = ""
    if torch.cuda.is_available():
        gpu = f" | cuda={torch.cuda.memory_allocated()/2**20:.0f}MiB allocated/{torch.cuda.memory_reserved()/2**20:.0f}MiB reserved"
    print(f"[{stamp}] {message}{gpu}", flush=True)


def select_threshold(labels, probabilities) -> float:
    """Exact O(n log n) F1 maximizer for the rule p >= threshold.

    Ties preserve the former exhaustive implementation's lowest-threshold rule.
    """
    labels = np.asarray(labels, dtype=np.int8).reshape(-1)
    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)
    if labels.size != probabilities.size or labels.size == 0:
        raise ValueError("labels/probabilities must be equally sized and non-empty")
    order = np.argsort(-probabilities, kind="stable")
    p = probabilities[order]; y = labels[order]
    ends = np.r_[np.flatnonzero(p[:-1] != p[1:]), p.size - 1]
    tp = np.cumsum(y)[ends].astype(float)
    predicted = (ends + 1).astype(float)
    fp = predicted - tp; fn = float(y.sum()) - tp
    denom = 2 * tp + fp + fn
    scores = np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom != 0)
    thresholds = p[ends]
    # Include the exhaustive implementation's explicit 0 and 1 candidates.
    extra_t = np.array([0.0, 1.0])
    extra_scores = np.array([f1_score(labels, probabilities >= t, zero_division=0) for t in extra_t])
    thresholds = np.r_[thresholds, extra_t]; scores = np.r_[scores, extra_scores]
    best = scores.max()
    return float(thresholds[scores == best].min())


def expected_calibration_error(labels, probabilities, bins=10) -> float:
    result = 0.0
    for lower, upper in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        mask = (probabilities >= lower) & (probabilities < (upper if upper < 1 else 1.000001))
        if mask.any(): result += abs(labels[mask].mean() - probabilities[mask].mean()) * mask.mean()
    return float(result)


def measure(labels, probabilities, threshold, cfg: Config) -> dict:
    predicted = (probabilities >= threshold).astype(int); tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {"ROC-AUC": float(roc_auc_score(labels, probabilities)), "PR-AUC": float(average_precision_score(labels, probabilities)),
        "Precision": float(precision_score(labels, predicted, zero_division=0)), "Recall": float(recall_score(labels, predicted, zero_division=0)),
        "F1": float(f1_score(labels, predicted, zero_division=0)), "selected_threshold": float(threshold),
        "ECE": expected_calibration_error(labels, probabilities, cfg.ece_bins), "Brier": float(brier_score_loss(labels, probabilities)),
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)}


def _hetero_predict(model, graph, device):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(graph.x_dict, graph.edge_index_dict)[0]).detach().cpu().numpy()


def train_proposed(sources, target_features, seed, cfg: Config, *, adaptive=False, use_domain_generalization=True, variant=None):
    """Train with source labels and an optional label-free target feature view."""
    variant = dict(variant or {}); fixed_threshold = variant.pop("fixed_threshold", False); device = _device(); set_seed(seed)
    metadata = next(iter(sources.values())).metadata()
    local_domains = {name: index for index, name in enumerate(sources)}
    target_domain = len(local_domains)
    n_domains = len(local_domains) + (1 if adaptive else 0)
    model = DGHeteroGNN(metadata, cfg, n_domains=max(2, n_domains), **variant).to(device); optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    focal, domain_loss = FocalLoss(cfg.focal_alpha, cfg.focal_gamma), nn.CrossEntropyLoss()
    splits = {name: transaction_split(graph, seed, cfg.validation_fraction) for name, graph in sources.items()}
    source_device = {name: _device_view(graph, device, include_labels=True) for name, graph in sources.items()}
    target_device = _device_view(target_features, device, include_labels=False) if adaptive else None
    epochs = cfg.smoke_epochs if cfg.run_mode == "SMOKE_TEST" else cfg.epochs
    started = time.perf_counter()
    for epoch in range(epochs):
        model.train(); optimizer.zero_grad(); total = 0.
        for name, graph in source_device.items():
            logits, domains = model(graph.x_dict, graph.edge_index_dict, cfg.grl_strength); train, _ = splits[name]
            total = total + focal(logits[train.to(device)], graph["transaction"].y[train.to(device)])
            if use_domain_generalization:
                total = total + cfg.domain_weight * domain_loss(domains, torch.full((domains.size(0),), local_domains[name], dtype=torch.long, device=device))
        if adaptive and use_domain_generalization:
            _, domains = model(target_device.x_dict, target_device.edge_index_dict, cfg.grl_strength)
            total = total + cfg.domain_weight * domain_loss(domains, torch.full((domains.size(0),), target_domain, dtype=torch.long, device=device))
        total.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.); optimizer.step()
        _progress(f"epoch {epoch+1}/{epochs} loss={float(total.detach()):.6f} elapsed={time.perf_counter()-started:.1f}s")
    yv, pv = [], []
    for name, graph in source_device.items():
        _, valid = splits[name]; p = _hetero_predict(model, graph, device); yv.append(graph["transaction"].y[valid].cpu().numpy()); pv.append(p[valid.numpy()])
    threshold = .5 if fixed_threshold else select_threshold(np.concatenate(yv), np.concatenate(pv))
    return model.cpu(), threshold, {"validation": "pooled source validation labels/probabilities", "adaptive": adaptive,
        "domain_generalization": use_domain_generalization, "local_domain_ids": local_domains}


def evaluate_hetero(model, target, device=None):
    """The only heterogeneous evaluation path allowed to read target labels."""
    device = device or _device(); model = model.to(device)
    view = _device_view(target, device, include_labels=False)
    p = _hetero_predict(model, view, device)
    y = target["transaction"].y.detach().cpu().numpy()
    return model.cpu(), y, p


def train_heterosage(sources, seed, cfg: Config):
    """Independent baseline trainer with fraud loss only."""
    device = _device(); set_seed(seed); model = HeteroGraphSAGE(next(iter(sources.values())).metadata(), cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    focal = FocalLoss(cfg.focal_alpha, cfg.focal_gamma)
    splits = {name: transaction_split(graph, seed, cfg.validation_fraction) for name, graph in sources.items()}
    views = {name: _device_view(graph, device, include_labels=True) for name, graph in sources.items()}
    epochs = cfg.smoke_epochs if cfg.run_mode == "SMOKE_TEST" else cfg.epochs
    for epoch in range(epochs):
        model.train(); optimizer.zero_grad(); loss = 0.
        for name, graph in views.items():
            train, _ = splits[name]; logits = model(graph.x_dict, graph.edge_index_dict)
            loss = loss + focal(logits[train.to(device)], graph["transaction"].y[train.to(device)])
        loss.backward(); optimizer.step(); _progress(f"HeteroGraphSAGE epoch {epoch+1}/{epochs} loss={float(loss.detach()):.6f}")
    model.eval(); yv=[]; pv=[]
    with torch.no_grad():
        for name, graph in views.items():
            _, valid = splits[name]; yv.append(graph["transaction"].y[valid.to(device)].cpu().numpy())
            pv.append(torch.sigmoid(model(graph.x_dict, graph.edge_index_dict))[valid.to(device)].cpu().numpy())
    return model.cpu(), select_threshold(np.concatenate(yv), np.concatenate(pv)), {"validation":"pooled source validation labels/probabilities", "domain_generalization":False}


def flatten_graph(graph: HeteroData) -> tuple[torch.Tensor, torch.Tensor, int]:
    """Documented homogeneous conversion: concatenate original nodes and relations only."""
    offsets, cursor = {}, 0
    features = []
    for node_type in ("transaction", "account", "merchant"):
        x = graph[node_type].x.float()
        offsets[node_type] = cursor; cursor += x.size(0)
        features.append(F.pad(x, (0, 128 - x.size(1))))
    edges = []
    for source, relation, target in graph.edge_types:
        # Preserve each archived semantic direction exactly once.
        edge = graph[(source, relation, target)].edge_index
        directed = torch.stack((edge[0] + offsets[source], edge[1] + offsets[target]))
        edges.append(directed)
    return torch.cat(features), torch.cat(edges, dim=1), graph["transaction"].num_nodes


def _train_feature_model(kind, sources, target, seed, cfg: Config):
    """Train MLP/XGBoost and homogeneous GNN baselines with exactly source splits."""
    set_seed(seed); split = {name: transaction_split(g, seed, cfg.validation_fraction) for name, g in sources.items()}
    if kind == "xgb":
        import xgboost as xgb
        tx = np.concatenate([g["transaction"].x[split[name][0]].numpy() for name, g in sources.items()]); y = np.concatenate([g["transaction"].y[split[name][0]].numpy() for name, g in sources.items()])
        model = xgb.XGBClassifier(n_estimators=20 if cfg.run_mode == "SMOKE_TEST" else 100, max_depth=6, learning_rate=.05, random_state=seed, n_jobs=1, eval_metric="logloss")
        model.fit(tx, y)
        val_y = np.concatenate([g["transaction"].y[split[name][1]].numpy() for name, g in sources.items()])
        val_p = np.concatenate([model.predict_proba(g["transaction"].x[split[name][1]].numpy())[:, 1] for name, g in sources.items()])
        return model, target["transaction"].y.numpy(), model.predict_proba(target["transaction"].x.numpy())[:, 1], select_threshold(val_y, val_p), {"validation":"pooled source validation labels/probabilities"}
    device = _device(); focal = FocalLoss(cfg.focal_alpha, cfg.focal_gamma)
    if kind == "mlp":
        model = MLP().to(device); optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
        for _ in range(cfg.smoke_epochs if cfg.run_mode == "SMOKE_TEST" else cfg.epochs):
            optimizer.zero_grad(); loss = 0.
            for name, g in sources.items():
                tr, _ = split[name]; x, y = g["transaction"].x.to(device), g["transaction"].y.to(device)
                loss = loss + focal(model(x)[tr.to(device)], y[tr.to(device)])
            loss.backward(); optimizer.step()
        with torch.no_grad():
            val_y = np.concatenate([g["transaction"].y[split[name][1]].numpy() for name, g in sources.items()])
            val_p = np.concatenate([torch.sigmoid(model(g["transaction"].x.to(device)))[split[name][1].to(device)].cpu().numpy() for name, g in sources.items()])
            target_p = torch.sigmoid(model(target["transaction"].x.to(device))).cpu().numpy()
        return model.cpu(), target["transaction"].y.numpy(), target_p, select_threshold(val_y, val_p), {"validation":"pooled source validation labels/probabilities"}
    model = HomogeneousGNN(kind, cfg.hidden).to(device); optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    flat = {name: tuple(x.to(device) if isinstance(x, torch.Tensor) else x for x in flatten_graph(g)) for name, g in sources.items()}
    for _ in range(cfg.smoke_epochs if cfg.run_mode == "SMOKE_TEST" else cfg.epochs):
        optimizer.zero_grad(); loss = 0.
        for name, g in sources.items():
            x, edge, n = flat[name]; tr, _ = split[name]
            loss = loss + focal(model(x, edge, n)[tr.to(device)], g["transaction"].y[tr].to(device))
        loss.backward(); optimizer.step()
    with torch.no_grad():
        val_y, val_p = [], []
        for name, g in sources.items():
            _, valid = split[name]; x, edge, n = flat[name]; val_y.append(g["transaction"].y[valid].numpy()); val_p.append(torch.sigmoid(model(x, edge, n))[valid.to(device)].cpu().numpy())
        x, edge, n = (v.to(device) if isinstance(v, torch.Tensor) else v for v in flatten_graph(target)); target_p = torch.sigmoid(model(x, edge, n)).cpu().numpy()
    return model.cpu(), target["transaction"].y.numpy(), target_p, select_threshold(np.concatenate(val_y), np.concatenate(val_p)), {"validation":"pooled source validation labels/probabilities"}


def _aggregate(rows: list[dict]) -> dict:
    keys = rows[0].keys()
    return {key: {"mean": float(np.mean([row[key] for row in rows])), "std": float(np.std([row[key] for row in rows], ddof=0))} for key in keys}


def _save_result(root: Path, scenario: str, model_name: str, seed: int, y, p, metrics, metadata, cfg: Config, model) -> str:
    token = f"{scenario}_{model_name.replace(' ','_')}_seed{seed}"
    prediction = root / "predictions" / f"{token}.npz"; prediction.parent.mkdir(parents=True, exist_ok=True)
    temp = prediction.with_suffix(".tmp.npz"); np.savez_compressed(temp, labels=y, probabilities=p, **{"fpr":roc_curve(y,p)[0],"tpr":roc_curve(y,p)[1],"precision":precision_recall_curve(y,p)[0],"recall":precision_recall_curve(y,p)[1]}); temp.replace(prediction)
    if model_name != "XGBoost": atomic_torch(root / "models" / f"{token}.pt", model.state_dict())
    record = {"scenario":scenario,"model":model_name,"seed":seed,"configuration":asdict(cfg),"threshold_provenance":metadata,"metrics":metrics,"prediction_artifact":str(prediction),"dataset_hashes":HASHES}
    atomic_json(root / "results" / f"{token}.json", record)
    return str(prediction)


def run_scenario(graphs: dict, artifacts: Path, scenario: str, source_names: list[str], target_name: str, model_name: str, cfg: Config, *, adaptive=False, variant=None) -> dict:
    records, paths = [], []
    seeds = cfg.seeds if cfg.run_mode == "FULL_PAPER_RUN" else (cfg.seeds[0],)
    for seed in seeds:
        token = f"{scenario}_{model_name.replace(' ','_')}_seed{seed}"
        checkpoint = artifacts / "results" / f"{token}.json"
        if checkpoint.exists():
            saved = json.loads(checkpoint.read_text(encoding="utf-8"))
            if saved.get("configuration") == asdict(cfg) and saved.get("dataset_hashes") == HASHES and Path(saved["prediction_artifact"]).exists():
                _progress(f"RESUME {scenario} | {model_name} | seed={seed}")
                records.append(saved["metrics"]); paths.append(saved["prediction_artifact"]); continue
        _progress(f"START {scenario} | {model_name} | seed={seed}")
        sources = {name: graphs[name] for name in source_names}; target = graphs[target_name]
        if cfg.run_mode == "SMOKE_TEST":
            sources = {name: _sample_graph(graph, seed + DOMAIN_ID[name], cfg.smoke_nodes) for name, graph in sources.items()}
            target = _sample_graph(target, seed + DOMAIN_ID[target_name], cfg.smoke_nodes)
        if model_name == "DG-Hetero-GNN":
            use_dg = not (scenario == "ABLATION_without_domain_generalization")
            model, threshold, provenance = train_proposed(sources, target, seed, cfg, adaptive=adaptive, use_domain_generalization=use_dg, variant=variant)
            model, y, p = evaluate_hetero(model, target)
        elif model_name == "HeteroGraphSAGE":
            model, threshold, provenance = train_heterosage(sources, seed, cfg)
            device = _device(); model = model.to(device); view = _device_view(target, device, include_labels=False)
            model.eval()
            with torch.no_grad(): p = torch.sigmoid(model(view.x_dict, view.edge_index_dict)).cpu().numpy()
            model = model.cpu(); y = target["transaction"].y.cpu().numpy()
        else: model, y, p, threshold, provenance = _train_feature_model({"GraphSAGE":"sage","GCN":"gcn","GAT":"gat","MLP":"mlp","XGBoost":"xgb"}[model_name], sources, target, seed, cfg)
        if variant and variant.get("fixed_threshold"): threshold = .5
        item = measure(y, p, threshold, cfg); records.append(item); paths.append(_save_result(artifacts, scenario, model_name, seed, y, p, item, provenance, cfg, model))
        _progress(f"DONE {scenario} | {model_name} | seed={seed} | F1={item['F1']:.6f}")
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    result = {"scenario":scenario,"model":model_name,"source_domains":source_names,"target_domain":target_name,"per_seed":records,"aggregate":_aggregate(records),"prediction_artifacts":paths,"dataset_hashes":HASHES,"run_mode":cfg.run_mode}
    atomic_json(artifacts / "results" / f"{scenario}_{model_name.replace(' ','_')}.json", result)
    return result


def render_outputs(artifacts: Path, results: list[dict]) -> None:
    """Write numerical CSVs and publication-sized figures from saved run results only."""
    import matplotlib.pyplot as plt
    tables, figures = artifacts / "tables", artifacts / "figures"; tables.mkdir(parents=True, exist_ok=True); figures.mkdir(parents=True, exist_ok=True)
    rows = []
    for result in results:
        rows.append({"scenario":result["scenario"],"model":result["model"], **{f"{k}_mean":v["mean"] for k,v in result["aggregate"].items()}, **{f"{k}_std":v["std"] for k,v in result["aggregate"].items()}})
    frame = pd.DataFrame(rows); frame.to_csv(tables / "all_results.csv", index=False)
    # Tables 2--5 are direct projections of per-scenario aggregate artifacts.
    lodo = frame[frame.scenario.str.startswith("LODO")]
    lodo.to_csv(tables / "table2_lodo_all_models.csv", index=False)
    lodo[lodo.model == "DG-Hetero-GNN"].to_csv(tables / "table3_lodo_dg_hetero_gnn.csv", index=False)
    frame[frame.scenario == "OOD"].to_csv(tables / "table4_ood_amlsim.csv", index=False)
    frame[frame.scenario.isin(["OOD", "ADAPTIVE_OOD"]) & (frame.model == "DG-Hetero-GNN")].to_csv(tables / "table5_ood_vs_adaptive.csv", index=False)
    for metric, filename in [("ROC-AUC","figure3a_lodo_roc_auc.png"),("PR-AUC","figure3b_lodo_pr_auc.png"),("F1","figure3c_lodo_f1.png")]:
        data = frame[frame.scenario.str.startswith("LODO")].copy()
        if data.empty: continue
        pivot = data.pivot(index="scenario", columns="model", values=f"{metric}_mean")
        pivot.plot(kind="bar", figsize=(10,5), color=["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"])
        plt.ylabel(metric); plt.title(f"LODO {metric} (computed artifacts)"); plt.tight_layout(); plt.savefig(figures / filename, dpi=300); plt.close()
    # Figures 2(a,b): all OOD models share one axes; source coordinates are saved.
    curve_rows=[]; pr_fig, pr_ax=plt.subplots(figsize=(7,5)); roc_fig, roc_ax=plt.subplots(figsize=(7,5))
    for result in results:
        if result["scenario"] != "OOD" or not result["prediction_artifacts"]: continue
        data = np.load(result["prediction_artifacts"][0]); model=result["model"]
        pr_ax.plot(data["recall"],data["precision"],label=model); roc_ax.plot(data["fpr"],data["tpr"],label=model)
        curve_rows.extend([{"model":model,"curve":"PR","x":float(x),"y":float(y)} for x,y in zip(data["recall"],data["precision"])])
        curve_rows.extend([{"model":model,"curve":"ROC","x":float(x),"y":float(y)} for x,y in zip(data["fpr"],data["tpr"])])
    if curve_rows:
        pd.DataFrame(curve_rows).to_csv(tables / "figure2ab_curve_coordinates.csv",index=False)
        pr_ax.set(xlabel="Recall",ylabel="Precision",title="OOD Precision-Recall"); pr_ax.legend(); pr_fig.tight_layout(); pr_fig.savefig(figures / "figure2a_ood_pr.png",dpi=300)
        roc_ax.plot([0,1],[0,1],"k--"); roc_ax.set(xlabel="False Positive Rate",ylabel="True Positive Rate",title="OOD ROC"); roc_ax.legend(); roc_fig.tight_layout(); roc_fig.savefig(figures / "figure2b_ood_roc.png",dpi=300)
    plt.close(pr_fig); plt.close(roc_fig)
    ood = frame[frame.scenario == "OOD"]
    if not ood.empty:
        # Figure 2(c): each baseline's first-seed confusion totals saved in result artifacts.
        cms, labels = [], []
        for result in results:
            if result["scenario"] == "OOD" and result["model"] != "DG-Hetero-GNN":
                seed = result["per_seed"][0]; cms.append(np.array([[seed["TN"], seed["FP"]], [seed["FN"], seed["TP"]]])); labels.append(result["model"])
        if cms:
            fig, axes = plt.subplots(1, len(cms), figsize=(3.2*len(cms), 3.2), constrained_layout=True); axes = np.atleast_1d(axes)
            for ax, cm, label in zip(axes, cms, labels):
                im=ax.imshow(cm, cmap="Blues"); ax.set_title(label); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
                for i in range(2):
                    for j in range(2): ax.text(j,i,str(cm[i,j]),ha="center",va="center")
            fig.colorbar(im, ax=axes.ravel().tolist(), shrink=.78, pad=.02); fig.savefig(figures / "figure2c_ood_confusion_heatmaps.png",dpi=300); plt.close(fig)
        calibration = frame[(frame.model=="DG-Hetero-GNN") & frame.scenario.isin(["LODO0","LODO1","LODO2","OOD","ADAPTIVE_OOD"])][["scenario","target_domain","ECE_mean","Brier_mean"]] if "target_domain" in frame else frame[(frame.model=="DG-Hetero-GNN") & frame.scenario.isin(["LODO0","LODO1","LODO2","OOD","ADAPTIVE_OOD"])][["scenario","ECE_mean","Brier_mean"]]
        calibration.to_csv(tables / "figure2e_calibration_by_dataset.csv",index=False)
        plt.figure(figsize=(8,4)); calibration.set_index("scenario")[["ECE_mean","Brier_mean"]].plot(kind="bar",ax=plt.gca(),color=["#0072B2", "#D55E00"]); plt.title("DG-Hetero-GNN calibration across evaluation datasets"); plt.tight_layout(); plt.savefig(figures / "figure2e_calibration.png",dpi=300); plt.close()
    comparison = frame[frame.scenario.isin(["OOD","ADAPTIVE_OOD"]) & (frame.model=="DG-Hetero-GNN")]
    if len(comparison) == 2:
        values=[]
        for _, row in comparison.iterrows(): values.append([[row.TN_mean,row.FP_mean],[row.FN_mean,row.TP_mean]])
        fig, axes=plt.subplots(1,2,figsize=(6,3))
        for ax, cm, (_, row) in zip(axes, values, comparison.iterrows()):
            cm = np.asarray(cm)
            ax.imshow(cm,cmap="Greens"); ax.set_title(row.scenario)
            for i in range(2):
                for j in range(2): ax.text(j,i,f"{cm[i,j]:.0f}",ha="center",va="center")
        fig.tight_layout(); fig.savefig(figures / "figure2d_ood_vs_adaptive.png",dpi=300); plt.close(fig)
    ablation = frame[frame.scenario.str.startswith("ABLATION_")]
    if not ablation.empty:
        plt.figure(figsize=(10,4)); plt.bar(ablation.scenario.str.replace("ABLATION_","",regex=False),ablation.F1_mean,color="#0072B2"); plt.xticks(rotation=25,ha="right"); plt.ylabel("F1"); plt.title("Figure 3(d): Ablation study"); plt.tight_layout(); plt.savefig(figures / "figure3d_ablation.png",dpi=300); plt.close()


def run_suite(root: Path | None = None, cfg: Config | None = None, *, run_id: str | None = None, implementation_id: str = "shared-reviewed-core") -> list[dict]:
    root = project_root(root); cfg = cfg or Config(run_mode=os.environ.get("DG_HETERO_RUN_MODE", "SMOKE_TEST"))
    run_id = run_id or os.environ.get("DG_HETERO_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    if not run_id.replace("-", "").replace("_", "").isalnum(): raise ValueError("run_id may contain only letters, digits, '-' and '_'")
    artifacts = root / "artifacts_runs" / implementation_id / cfg.run_mode / run_id
    graphs, table = load_all(root)
    try:
        import xgboost
        xgboost_version = xgboost.__version__
    except ImportError: xgboost_version = None
    import matplotlib
    manifest = {"status":"running","started_at":datetime.now(timezone.utc).isoformat(),"run_id":run_id,"implementation_id":implementation_id,
        "artifact_root":str(artifacts),"configuration":asdict(cfg),"dataset_hashes":HASHES,"python":sys.version,"torch":torch.__version__,
        "torch_geometric":torch_geometric.__version__,"numpy":np.__version__,"pandas":pd.__version__,"scikit_learn":sklearn.__version__,
        "xgboost":xgboost_version,"matplotlib":matplotlib.__version__,"cuda":torch.version.cuda,"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
    existing = artifacts / "manifest.json"
    if existing.exists():
        prior = json.loads(existing.read_text(encoding="utf-8"))
        if prior.get("configuration") != asdict(cfg) or prior.get("dataset_hashes") != HASHES:
            raise RuntimeError(f"run scope {artifacts} belongs to a different configuration or dataset set")
    atomic_json(existing, manifest); atomic_json(artifacts / "graph_statistics.json", table.to_dict(orient="records"))
    _progress(f"RUN {run_id} | {implementation_id} | {cfg.run_mode} | artifacts={artifacts}")
    (artifacts / "tables").mkdir(parents=True, exist_ok=True); table.to_csv(artifacts / "tables" / "table1_graph_statistics.csv", index=False)
    try:
        output = []
        models = ["DG-Hetero-GNN","XGBoost","MLP","GCN","GraphSAGE","GAT","HeteroGraphSAGE"]
        for scenario, (sources, target) in LODO.items():
            for model in models: output.append(run_scenario(graphs, artifacts, scenario, sources, target, model, cfg))
        for model in models: output.append(run_scenario(graphs, artifacts, "OOD", ["elliptic","ieee","dgraphfin"], "amlsim", model, cfg))
        output.append(run_scenario(graphs, artifacts, "ADAPTIVE_OOD", ["elliptic","ieee","dgraphfin"], "amlsim", "DG-Hetero-GNN", cfg, adaptive=True))
        ablations = {"full":{}, "without_type_projection":{"type_projection":False}, "without_heterogeneous_message_passing":{"message_passing":False}, "without_domain_generalization":{}, "without_deep_classifier":{"deep_head":False}, "without_threshold_optimization":{"fixed_threshold":True}}
        for label, variant in ablations.items(): output.append(run_scenario(graphs, artifacts, "ABLATION_"+label, ["elliptic","ieee","dgraphfin"], "amlsim", "DG-Hetero-GNN", cfg, adaptive=label == "full", variant=variant))
        render_outputs(artifacts, output)
        manifest.update(status="complete", completed_at=datetime.now(timezone.utc).isoformat(), aggregate_results=len(output))
        atomic_json(existing, manifest); _progress(f"COMPLETE run={run_id} results={len(output)}")
        return output
    except Exception as error:
        manifest.update(status="failed", failed_at=datetime.now(timezone.utc).isoformat(), error=repr(error))
        atomic_json(existing, manifest); _progress(f"FAILED run={run_id}: {error!r}"); raise


## Artifact-backed experiment suite

Progress is flushed at every scenario/model/seed and epoch. A stable run ID resumes atomic per-seed checkpoints; different modes and notebook implementations cannot share artifacts.

In [ ]:
ROOT = project_root(Path.cwd())
RUN_MODE = os.environ.get("DG_HETERO_RUN_MODE", "SMOKE_TEST")
RUN_ID = os.environ.get("DG_HETERO_RUN_ID", "interactive-smoke")
IMPLEMENTATION_ID = "clean-notebook" if "Clean" in "DG-Hetero-GNN Paper Minimal-Diff Recovery" else "minimal-diff-notebook"
CORE_CONFIG = Config(run_mode=RUN_MODE)
SUITE_RESULTS = run_suite(ROOT, CORE_CONFIG, run_id=RUN_ID, implementation_id=IMPLEMENTATION_ID)
summary = pd.DataFrame([{"scenario": r["scenario"], "model": r["model"], "F1 mean": r["aggregate"]["F1"]["mean"], "run mode": r["run_mode"]} for r in SUITE_RESULTS])
display(summary)
print("Completed run", RUN_ID, "for", IMPLEMENTATION_ID)
